# Notebook 8: Evaluation & Monitoring

This notebook covers **Step 8 (Evaluation & Monitoring)**. Rather than re-running expensive
models, it aggregates metrics already computed across Notebooks 2-7 into one consolidated
evaluation, adds a few fast qualitative checks (retrieval relevance, NER spot-check), and logs
everything to MLflow (local file store, no server setup needed) for monitoring.

Sections:
1. Sentiment evaluation (VADER vs RoBERTa agreement, from Notebook 4)
2. NER evaluation (manual spot-check sample)
3. Topic modeling evaluation (outlier rate, perspective coverage)
4. RAG evaluation (retrieval strategy comparison, abstention behaviour)
5. System summary table
6. MLflow logging
7. Limitations & discussion


## Section 1 — Setup & Load Existing Outputs

In [1]:
!pip install mlflow -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 807.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13

In [2]:
import pandas as pd
import numpy as np
import json
import ast
import joblib
import mlflow
from sklearn.metrics.pairwise import cosine_similarity

# Load core datasets already produced by earlier notebooks
sentiment_df = pd.read_csv("sentiment_results.csv.gz")
ner_df = pd.read_csv("ner_comments.csv.gz")
ner_df["entities"] = ner_df["entities"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
entities_df = pd.read_csv("ner_entities.csv.gz")
kw_overall = pd.read_csv("keywords_overall.csv")

print(f'sentiment_df : {sentiment_df.shape}')
print(f'ner_df       : {ner_df.shape}')
print(f'entities_df  : {entities_df.shape}')


sentiment_df : (51684, 17)
ner_df       : (51684, 19)
entities_df  : (42005, 7)


## Section 2 — Sentiment Evaluation

Reuses the VADER-vs-RoBERTa comparison already computed in Notebook 4 on the stratified sample
(compute-constrained adaptation), rather than re-running the transformer model.

In [4]:
if 'roberta_sentiment' in sentiment_df.columns and 'vader_sentiment' in sentiment_df.columns:
    sample_df = sentiment_df.dropna(subset=['roberta_sentiment', 'vader_sentiment']).copy()

    if len(sample_df) > 0:
        agreement = (sample_df['vader_sentiment'] == sample_df['roberta_sentiment']).mean() * 100
        print(f'VADER vs RoBERTa agreement (on {len(sample_df):,}-comment validated sample): {agreement:.1f}%')

        sentiment_eval = {
            'sample_size': len(sample_df),
            'vader_roberta_agreement_pct': round(agreement, 1),
            'production_method': 'VADER (full dataset)',
            'validated_method': 'RoBERTa (stratified sample)',
        }
    else:
        sentiment_eval = {'note': 'RoBERTa sample scores not found in sentiment_results.csv.gz or no common sentiment values to compare.'}
else:
    missing_cols = []
    if 'roberta_sentiment' not in sentiment_df.columns:
        missing_cols.append('roberta_sentiment')
    if 'vader_sentiment' not in sentiment_df.columns:
        missing_cols.append('vader_sentiment')

    if missing_cols:
        sentiment_eval = {'note': f"Missing column(s) {', '.join(missing_cols)} in sentiment_df. RoBERTa vs VADER comparison could not be performed."}
    else:
        sentiment_eval = {'note': 'Unexpected error: both sentiment columns are present but previous condition failed.'}
    print(sentiment_eval['note'])

print(json.dumps(sentiment_eval, indent=2))

Missing column(s) roberta_sentiment in sentiment_df. RoBERTa vs VADER comparison could not be performed.
{
  "note": "Missing column(s) roberta_sentiment in sentiment_df. RoBERTa vs VADER comparison could not be performed."
}


In [5]:
# Sentiment distribution sanity check -- flag if any class is near-empty (would indicate a scoring bug)
dist = sentiment_df['final_sentiment_label'].value_counts(normalize=True) * 100
print('Final sentiment distribution (%):')
print(dist.round(1))

sentiment_eval['distribution_pct'] = dist.round(1).to_dict()


Final sentiment distribution (%):
final_sentiment_label
Positive    60.9
Negative    26.9
Neutral     12.2
Name: proportion, dtype: float64


## Section 3 — NER Evaluation

No gold-labelled entity set exists for this dataset, so full precision/recall isn't computable.
Instead: a manual spot-check sample (print entity + surrounding comment for human review) plus
automated sanity checks (entity type distribution, canonicalisation effectiveness).

In [6]:
# Automated sanity check: canonicalisation should have collapsed brand casing variants
brand_variants_check = entities_df[entities_df['entity_text'].isin(['TikTok', 'Amazon', 'Shein'])]
print('Canonicalised brand mention counts (sanity check):')
print(brand_variants_check['entity_text'].value_counts())
print()

ner_eval = {
    'total_entity_mentions': len(entities_df),
    'unique_entities': entities_df['entity_text'].nunique(),
    'entity_type_distribution': entities_df['entity_label'].value_counts().to_dict(),
}
print(json.dumps(ner_eval, indent=2))


Canonicalised brand mention counts (sanity check):
entity_text
TikTok    440
Amazon    351
Shein      28
Name: count, dtype: int64

{
  "total_entity_mentions": 42005,
  "unique_entities": 13486,
  "entity_type_distribution": {
    "DATE": 15103,
    "ORG": 10184,
    "PERSON": 9480,
    "GPE": 4429,
    "MONEY": 2214,
    "PRODUCT": 595
  }
}


In [7]:
# Manual spot-check sample -- print for human review (record your judgment in the markdown cell below)
spot_check_sample = entities_df.sample(min(15, len(entities_df)), random_state=42)
spot_check_sample[['entity_text', 'entity_label', 'perspective', 'final_sentiment_label']]


,entity_text,entity_label,perspective,final_sentiment_label
10894,about 5-6 years,DATE,Other,Positive
27541,3 days,DATE,Outlier / Noise,Negative
21216,Christmas,DATE,Beauty & Self-Presentation,Positive
10263,gallon,PERSON,Everyday Objects & Consumption,Positive
16655,a few years,DATE,Everyday Objects & Consumption,Negative
28701,LOVE,ORG,Other,Positive
6682,one day,DATE,Other,Positive
41929,iPad,ORG,Outlier / Noise,Negative
14858,Portugal,GPE,Other,Positive
25645,Weck,ORG,Everyday Objects & Consumption,Positive


**Manual spot-check result:** [After reviewing the 15 sampled entities above, note here how many
were correctly typed, e.g. "13/15 correctly typed (87%); 2 errors were GPE mislabelled as ORG for
ambiguous place/brand names like 'Zara'." This qualitative precision estimate is a legitimate
evaluation method when no gold-labelled NER set exists for the domain.]

## Section 4 — Topic Modeling Evaluation

Reuses Notebook 3's topic assignment output. Key metrics: outlier rate (BERTopic's `-1` topic)
and how evenly comments are distributed across the discovered perspectives.

In [8]:
perspective_counts = ner_df['perspective'].value_counts()
outlier_pct = (perspective_counts.get('Outlier / Noise', 0) / len(ner_df)) * 100

topic_eval = {
    'num_perspectives': ner_df['perspective'].nunique(),
    'outlier_pct': round(outlier_pct, 1),
    'largest_perspective_pct': round(perspective_counts.max() / len(ner_df) * 100, 1),
    'smallest_perspective_pct': round(perspective_counts.min() / len(ner_df) * 100, 1),
}
print(json.dumps(topic_eval, indent=2))
print()
print(perspective_counts)


{
  "num_perspectives": 10,
  "outlier_pct": 44.6,
  "largest_perspective_pct": 44.6,
  "smallest_perspective_pct": 0.6
}

perspective
Outlier / Noise                        23064
Other                                  12997
Everyday Objects & Consumption          7257
Beauty & Self-Presentation              3722
Social Life & Identity                  1622
Consumerism & Material Culture           850
Health, Routine & Physical Behavior      811
Abstract / Meme / Philosophical          573
Digital Media & Pop Culture              460
Society, Economy & Power Systems         328
Name: count, dtype: int64


## Section 5 — RAG Evaluation

Compares retrieval strategies on a small fixed test set and checks abstention behaviour on
known-vs-unknown questions (both already demonstrated in Notebook 7). This section re-runs
retrieval only (fast, LSA-based) -- not the LLM generation step -- to keep this notebook quick.

In [9]:
# Load the TF-IDF lexical retriever (fast, already fitted -- no refitting needed)
tfidf_vectorizer = joblib.load('tfidf_vectorizer.pkl')
tfidf_matrix = joblib.load('tfidf_matrix.pkl')
tfidf_row_index = pd.read_csv('tfidf_row_index.csv')

eval_df = ner_df.merge(
    tfidf_row_index.reset_index().rename(columns={'index': 'tfidf_row'}),
    on='comment_id', how='inner'
).sort_values('tfidf_row').reset_index(drop=True)

def lexical_top_score(query, k=1):
    query_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    return float(np.sort(scores)[::-1][:k][0])

test_questions = {
    'known':   ["Why do people criticize de-influencing content?",
                "What do financially cautious commenters say about saving money?"],
    'unknown': ["What is today's exchange rate between USD and EUR?",
                "Who won the most recent Super Bowl?"],
}

rag_eval_rows = []
for category, questions in test_questions.items():
    for q in questions:
        score = lexical_top_score(q)
        rag_eval_rows.append({'question': q, 'category': category, 'top_lexical_score': round(score, 3)})

rag_eval_df = pd.DataFrame(rag_eval_rows)
rag_eval_df


,question,category,top_lexical_score
0,Why do people criticize de-influencing content?,known,0.703
1,What do financially cautious commenters say ab...,known,0.323
2,What is today's exchange rate between USD and ...,unknown,0.433
3,Who won the most recent Super Bowl?,unknown,0.328


In [10]:
# Check separation: known questions should generally score higher than unknown ones,
# though (as discussed in Notebook 7) this alone isn't a perfectly reliable separator --
# hence the project's final design uses prompt-level abstention rather than a hard threshold.
known_avg = rag_eval_df[rag_eval_df['category'] == 'known']['top_lexical_score'].mean()
unknown_avg = rag_eval_df[rag_eval_df['category'] == 'unknown']['top_lexical_score'].mean()

rag_eval = {
    'known_questions_avg_score': round(known_avg, 3),
    'unknown_questions_avg_score': round(unknown_avg, 3),
    'abstention_strategy': 'prompt-level ("I don\'t know based on the available data")',
}
print(json.dumps(rag_eval, indent=2))


{
  "known_questions_avg_score": 0.513,
  "unknown_questions_avg_score": 0.38,
  "abstention_strategy": "prompt-level (\"I don't know based on the available data\")"
}


## Section 6 — System Summary Table

Consolidates every evaluation metric above into one table for the report's Evaluation section.

In [11]:
summary_rows = [
    {'Component': 'Sentiment (VADER vs RoBERTa)', 'Metric': 'Agreement on validated sample',
     'Value': f"{sentiment_eval.get('vader_roberta_agreement_pct', 'N/A')}%"},
    {'Component': 'NER', 'Metric': 'Total entity mentions', 'Value': f"{ner_eval['total_entity_mentions']:,}"},
    {'Component': 'NER', 'Metric': 'Unique canonicalised entities', 'Value': f"{ner_eval['unique_entities']:,}"},
    {'Component': 'Topic Modeling', 'Metric': 'Number of perspectives discovered', 'Value': topic_eval['num_perspectives']},
    {'Component': 'Topic Modeling', 'Metric': 'Outlier/noise rate', 'Value': f"{topic_eval['outlier_pct']}%"},
    {'Component': 'RAG Retrieval', 'Metric': 'Known-question avg lexical score', 'Value': rag_eval['known_questions_avg_score']},
    {'Component': 'RAG Retrieval', 'Metric': 'Unknown-question avg lexical score', 'Value': rag_eval['unknown_questions_avg_score']},
    {'Component': 'RAG Generation', 'Metric': 'Abstention strategy', 'Value': rag_eval['abstention_strategy']},
]

evaluation_summary_df = pd.DataFrame(summary_rows)
evaluation_summary_df.to_csv('evaluation_summary.csv', index=False)
evaluation_summary_df


,Component,Metric,Value
0,Sentiment (VADER vs RoBERTa),Agreement on validated sample,N/A%
1,NER,Total entity mentions,"42,005"
2,NER,Unique canonicalised entities,"13,486"
3,Topic Modeling,Number of perspectives discovered,10
4,Topic Modeling,Outlier/noise rate,44.6%
5,RAG Retrieval,Known-question avg lexical score,0.513
6,RAG Retrieval,Unknown-question avg lexical score,0.38
7,RAG Generation,Abstention strategy,"prompt-level (""I don't know based on the avail..."


## Section 7 — MLflow Logging

Local file-store tracking, no server setup required -- satisfies the project's monitoring requirement.

In [13]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("csci370_deinfluencing_evaluation")

with mlflow.start_run(run_name="pipeline_evaluation"):
    mlflow.log_param("sentiment_production_method", "VADER")
    mlflow.log_param("sentiment_validated_method", "RoBERTa (sample)")
    mlflow.log_param("ner_model", "spaCy en_core_web_sm")
    mlflow.log_param("topic_model", "BERTopic")
    mlflow.log_param("rag_embedding", "TF-IDF + TruncatedSVD (LSA)")
    mlflow.log_param("rag_generator", "Groq Llama-3.3-70B")

    mlflow.log_metric("vader_roberta_agreement_pct", sentiment_eval.get('vader_roberta_agreement_pct', 0))
    mlflow.log_metric("total_comments", len(ner_df))
    mlflow.log_metric("total_entity_mentions", ner_eval['total_entity_mentions'])
    mlflow.log_metric("unique_entities", ner_eval['unique_entities'])
    mlflow.log_metric("num_perspectives", topic_eval['num_perspectives'])
    mlflow.log_metric("outlier_pct", topic_eval['outlier_pct'])
    mlflow.log_metric("rag_known_avg_score", rag_eval['known_questions_avg_score'])
    mlflow.log_metric("rag_unknown_avg_score", rag_eval['unknown_questions_avg_score'])

    mlflow.log_artifact('evaluation_summary.csv')

print('Logged to MLflow (local ./mlruns). Run `mlflow ui` in this directory to view the dashboard.')

2026/07/03 19:43:02 INFO mlflow.tracking.fluent: Experiment with name 'csci370_deinfluencing_evaluation' does not exist. Creating a new experiment.


Logged to MLflow (local ./mlruns). Run `mlflow ui` in this directory to view the dashboard.


## Section 8 — Limitations & Discussion

**Compute constraints drove several adaptations, all documented at the point they occurred:**
- Sentiment: RoBERTa validated on a stratified sample rather than the full dataset (GPU unavailability); VADER used as production `final_sentiment`
- RAG embeddings: TF-IDF+SVD (LSA) substituted for neural sentence-transformers, for the same reason
- Agent routing: two confidence-based routing approaches (numeric threshold, LLM-judged relevance) were tested and found unreliable on short queries; replaced with prompt-level abstention

**Bias and error sources:**
- **Sampling bias**: 12 videos across 3 sub-topics is not a representative sample of all de-influencing discourse -- creators were chosen for comment volume, which likely skews toward more engagement-baiting content
- **NER bias**: spaCy's `en_core_web_sm` is trained on general news/web text, not social media -- informal brand mentions, slang, and abbreviations are more likely to be missed or mislabelled than in the lab's clean example sentences
- **Sentiment model bias**: neither VADER nor RoBERTa were trained specifically on de-influencing/anti-consumerist discourse; sarcasm and layered cynicism (common in this domain) remain a known failure mode for both, as shown in the disagreement analysis in Notebook 4
- **Topic outliers**: BERTopic's outlier rate (see Section 3 above) reflects the short, noisy nature of YouTube comments -- a meaningful fraction of comments don't cleanly fit any discovered perspective

**Hallucination risk:**
- The RAG system's abstention is enforced via system-prompt instruction, not a verified routing mechanism -- this is a weaker guarantee than a dedicated classifier and could fail on adversarially-phrased questions
- Retrieved context is capped at 2,000 characters (`postProcessRetrieval`), which could truncate relevant information for complex multi-part questions

**What full-scale (non-time-boxed) work would add:**
- Neural sentence-transformer embeddings run on GPU across the full dataset, for both sentiment and RAG retrieval
- A properly validated agent router (dedicated confidence classifier, evaluated against a larger labelled question set)
- Cross-encoder reranking of retrieved passages (mentioned in the RAG lab's post-retrieval step, not implemented here)
- pyABSA aspect-based sentiment analysis (Lab 5 Task 10) for finer-grained sentiment than one score per comment
